# AI ANALYST LAB

![](../_img/Ghost_TheSyntheticBanner.png)

### A Hands-on Course on AI for Data Analysts
## Session 01: Demand and operations analytics for a bike-sharing business

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This notebook accompanies the **AI Analyst LAB** course. Welcome to Session 01.

### Lecturer

[Goran S. Milovanović, PhD, DataKolektiv, Chief Scientist & Owner](https://www.linkedin.com/in/gmilovanovic/)

***
### What we will do today

This is your first real analyst's session. You will load a real-world dataset, get a feel for what it contains, summarize it numerically, visualize it, learn how statisticians think about uncertainty, and finally use Claude through Python to help you communicate what you found.

We will work through these sections, in this order:

| Section | What happens |
|---|---|
| 1.1 | The business case — what is being asked of you |
| 1.2 | Meet your Session Tutor (Claude Project) |
| 1.3 | Setup — imports and loading the data |
| 1.4 | First look — what is actually in the file? |
| 1.5 | Descriptive statistics — the analyst's starter toolkit |
| 1.6 | Visualization — see the data |
| 1.7 | Three "stories" about how data arises (Normal, Binomial, Poisson) |
| 1.8 | Sample versus population |
| 1.9 | The sampling distribution of the mean (Central Limit Theorem) |
| 1.10 | Standard deviation versus standard error |
| 1.11 | Our first Claude API call from Python |
| 1.12 | Stakeholder memo — translating analysis into a decision |
| 1.13 | References — what to study to deepen this session |

A few notes before you start:

- **Run the cells in order.** Each section builds on the one before it.
- **It is fine to be slow.** The notebook is intentionally narrative — read the explanations, do not just run the code.
- **Every line of code has a comment above it** explaining what that line does. Read the comments — they are your in-line textbook.
- **Use your Session 01 Tutor** (Claude Project) when something is unclear. The notebook *computes*; the tutor *explains*. The two work together — see Section 1.2 below.
- **Mathematics notation will appear**, but every symbol will be introduced and every formula will be accompanied by a tiny example. We never use a symbol without first saying out loud what it stands for.

The 5-step professional rhythm — *clarify the question → validate the data → visualize → apply a method → interpret in business language* — is repeated throughout the course. Notice it as it happens.

***
## 1.1 The business case

Imagine you have just joined **CityCycle**, a mid-sized urban bike-sharing operator. It is your first week, and your manager hands you a CSV file with two years of hourly rental records.

> *"Leadership wants two things by Friday. First, what does demand actually look like — when are we busy, when are we quiet? Second, how confident are we in the numbers we report? A one-page memo with recommendations."*

You sit down at your laptop. The CSV is open in front of you. This notebook is your week of work.

By the end of this session you will have:

1. **Loaded** a real dataset and looked at it carefully before doing anything else.
2. **Described** what is "typical" using a handful of plain numbers (mean, median, spread).
3. **Visualized** the data with an analyst's starter plots (histograms, boxplots, demand curves).
4. **Recognized** which kind of "story" the data is telling (Normal, Binomial, or Poisson).
5. **Reasoned honestly about uncertainty** using the **sampling distribution of the mean** and the **Central Limit Theorem (CLT)**.
6. **Made your first Claude API call** from Python to help you write up the results.
7. **Handed leadership a one-page memo** that answers their two questions.

Two threads weave through this whole notebook:

- **What is happening in the data?** — sections 1.3 through 1.7.
- **How sure are we?** — sections 1.8 through 1.10.

The memo in section 1.12 brings both threads together. Let's start.

***
## 1.2 Meet your Session Tutor (Claude Project)

Before you continue, you should have set up your **Session 01 Tutor** — a Claude Project configured specifically to teach you the statistics ideas behind this notebook in a gentle, beginner-friendly way.

If you have not done this yet, open the file **[`_tutors/TutorProjectCreation.md`](../_tutors/TutorProjectCreation.md)** in this repository and follow the step-by-step instructions. It takes about five minutes. Come back here when you are done.

Throughout this notebook, when something feels confusing, **paste your question into the tutor**. The notebook is here to walk you through the analysis. The tutor is here to slow down on the statistics whenever you need.

Some example questions you might paste into the tutor as you work through this notebook:

- *"What does it mean for a number to be a 'mean' vs. a 'median'? Could you walk me through it with a small example?"*
- *"Why does the histogram of our bike rentals lean to the right?"*
- *"Standard deviation vs. standard error — what is the difference, in plain English?"*
- *"I am looking at this `describe()` output. Can you help me read it for an operations manager?"*

Keep the tutor open in a second browser tab as you work through the rest of this notebook.

***
## 1.3 Setup — imports and loading the data

Every analysis begins the same way: **import the tools we need**, **load the data**, and **confirm the data loaded**. Let's walk through each piece.

### What is a library?

Python by itself can do many things, but the heavy lifting in data analysis is done by *libraries* — collections of useful, pre-written code that someone else maintains and that we can import into our notebook. The four libraries we will use today are:

- **`pandas`** — gives us the `DataFrame`, a spreadsheet-like table of data. Almost everything we do today is a `pandas` operation. By convention we shorten the name to `pd`.
- **`numpy`** — provides fast arrays of numbers and mathematical functions. By convention we shorten the name to `np`. `pandas` itself uses `numpy` under the hood, but we will also use it directly later for simulations.
- **`matplotlib.pyplot`** — the foundational plotting library in Python. By convention `plt`.
- **`seaborn`** — a friendlier statistical-plotting library built on top of `matplotlib`. By convention `sns`.

The line `%matplotlib inline` is a Jupyter-only instruction that tells the notebook *"show the plots directly here in the notebook, not in a separate window."*

Run the cell below to import everything.

In [ ]:
# Import pandas under the short alias `pd`; pandas gives us the DataFrame (tables of data).
import pandas as pd

# Import numpy under the alias `np`; numpy provides fast numeric arrays and math functions.
import numpy as np

# Import matplotlib's plotting module as `plt`; this is the foundational plotting library.
import matplotlib.pyplot as plt

# Import seaborn as `sns`; seaborn provides higher-level statistical plots on top of matplotlib.
import seaborn as sns

# Jupyter magic that tells the notebook to display plots inline (directly in the notebook).
%matplotlib inline

# (Cosmetic) Set seaborn's default style to "whitegrid" — clean, readable plots with light gridlines.
sns.set_theme(style="whitegrid")

# Print a confirmation message so we know the imports succeeded.
print("Libraries imported successfully.")

### Loading the data

Now we load the dataset.

The bike-sharing data lives at `_data/bike_sharing/hour.csv` *inside the repo*. This notebook is at `AI_AnalystLAB/Session01/AI_AnalystLAB01.ipynb`, so to reach the data we go **up one folder** (`..`, which takes us to the repo root `AI_AnalystLAB/`) and then **into `_data/bike_sharing/`**:

- `..` means "the folder above me" — from `Session01/`, the folder above is the repo root `AI_AnalystLAB/`.
- `../_data/bike_sharing/hour.csv` is therefore the full relative path to the file.

> **Why this exact path?** When you run a cell, the Python kernel's *current working directory* is the folder this notebook lives in (`Session01/`). All relative paths are resolved from there.

### What is a DataFrame?

The function `pd.read_csv(...)` reads a CSV file and returns a **`DataFrame`** — a table with rows and columns, very much like a spreadsheet, but accessible from Python. Each row of the DataFrame is one **observation** (one record), and each column is one **variable** (one measurement type).

We store the result in a variable named `df`. This is just convention; the name has no special meaning.

In [ ]:
# Read the CSV file from disk into a pandas DataFrame; we assign the table to the variable `df`.
df = pd.read_csv("../_data/bike_sharing/hour.csv")

# Print the Python type of `df` — we expect "DataFrame" if the load worked.
print("Type of df:", type(df).__name__)

# Print the number of rows in the DataFrame so we can confirm the file's size.
print("Rows loaded:", len(df))

**Expected output.** You should see:

```
Type of df: DataFrame
Rows loaded: 17379
```

If you instead see a `FileNotFoundError`, your notebook is not where it should be — make sure you are running `AI_AnalystLAB/Session01/AI_AnalystLAB01.ipynb` and that the `_data/bike_sharing/hour.csv` file actually exists alongside the repo.

> **Mini-recap.** We imported the tools we will use today (`pandas`, `numpy`, `matplotlib`, `seaborn`), and loaded the bike-sharing data into a `DataFrame` called `df`. From here on, every operation we do is on `df`.

***
## 1.4 First look — what is actually in the file?

Before anyone trusts your analysis, they should trust your data. The very first thing every analyst does with a new dataset is what we call a **sanity check** — confirm that what is in the file matches what you were told is in the file. Four quick checks make up the standard sanity check.

### Check 1: shape — how big is the table?

A `DataFrame` has a `.shape` attribute that tells you `(number_of_rows, number_of_columns)`.

In [ ]:
# `.shape` returns a (rows, columns) tuple — a quick way to confirm the table's size.
df.shape

You should see `(17379, 17)` — about 17 thousand rows and 17 columns. That matches "two years of hourly data" because $2 \times 365 \times 24 = 17{,}520$ — close enough (a few hours are missing in the source data).

### Check 2: head — what does a row look like?

The method `.head()` shows you the first few rows of the DataFrame, so you can see the actual structure.

In [ ]:
# `.head()` returns the first 5 rows of the DataFrame so we can see its structure.
df.head()

Each row is one **hour** in one specific day. You can see columns for the date, the hour, the season, weather conditions, and counts of bike rentals. We will look at every column in a moment.

### Check 3: dtypes — what kind of value is in each column?

A column can hold integers, decimals, dates, or text. `.dtypes` tells you what kind each column is.

In [ ]:
# `.dtypes` returns the data type stored in each column (int64, float64, object/text, etc.).
df.dtypes

Most columns are `int64` (whole numbers) or `float64` (decimal numbers). The `dteday` column is stored as `object` (text). For our purposes today, this is all fine — we will not need to parse dates.

### Check 4: missing values

Missing values are silent killers in analysis — they often hide bugs. `df.isna()` gives a DataFrame of `True`/`False` flags (`True` means "missing"); summing across each column tells us how many missing values per column.

In [ ]:
# Chain two operations: `.isna()` flags missing cells as True; `.sum()` counts True per column.
df.isna().sum()

All zeros. Excellent — the dataset is clean. Real-world data is almost never this clean, but the UCI bike-sharing dataset is curated.

### Plain-English column dictionary

Here is what each column actually means. Keep this table nearby — we will refer to it throughout the notebook.

| Column | Meaning |
|---|---|
| `instant` | Row index (1, 2, 3, …). Ignore it. |
| `dteday` | Date in `YYYY-MM-DD` form. |
| `season` | 1 = winter, 2 = spring, 3 = summer, 4 = fall. |
| `yr` | 0 = 2011, 1 = 2012. |
| `mnth` | Month (1–12). |
| `hr` | Hour of day (0–23). |
| `holiday` | 1 if a public holiday, 0 otherwise. |
| `weekday` | 0 = Sunday, 1 = Monday, …, 6 = Saturday. |
| `workingday` | 1 if a regular working day, 0 if weekend or holiday. |
| `weathersit` | 1 = clear, 2 = misty/cloudy, 3 = light snow or rain, 4 = heavy weather. |
| `temp` | Air temperature, normalized to 0–1. |
| `atemp` | "Feels-like" temperature, normalized to 0–1. |
| `hum` | Humidity, normalized to 0–1. |
| `windspeed` | Wind speed, normalized to 0–1. |
| `casual` | Number of casual (non-subscriber) bike rentals in that hour. |
| `registered` | Number of registered (subscriber) bike rentals in that hour. |
| **`cnt`** | **Total rentals in that hour** (= `casual` + `registered`). **This is the column we care about most.** |

The single most important column for our analysis is **`cnt`** — total rentals per hour. Everything we compute about "demand" today will be about this column.

> **Mini-recap.** We confirmed the dataset has 17,379 hours and 17 columns, no missing values, and we know what every column means. We are ready to summarize it.

***
## 1.5 Descriptive statistics — the analyst's starter toolkit

Now we ask the first real business question:

> **"What is a typical hour of demand at CityCycle?"**

To answer it, we summarize the `cnt` column with a small number of meaningful numbers. The tools for that are called **descriptive statistics**. In this section we introduce each concept formally — with notation, a tiny worked example, and the practical interpretation — before computing it on the bike-sharing data.

### A tiny example we will reuse

For every concept below, we will reuse the same little eight-number dataset to ground the math:

$$2,\ 4,\ 4,\ 4,\ 5,\ 5,\ 7,\ 9$$

So whenever you see *"In the tiny example…"*, picture those eight numbers.

### 1.5.1 The mean (also called the average)

The **mean** is the most familiar way to summarize "the typical value" in a set of numbers. Most people use the word *"average"* to mean exactly this.

#### Notation and formula

Given $n$ numbers $x_1, x_2, x_3, \ldots, x_n$, the **sample mean** is denoted $\bar{x}$ (read *"x-bar"*) and computed as:

$$\bar{x} \;=\; \frac{1}{n} \sum_{i=1}^{n} x_i$$

Let's read every symbol:

- $\bar{x}$ ("x-bar") — the mean we are computing.
- $n$ — the number of values we have.
- $x_i$ — the $i$-th value, where $i$ counts from 1 up to $n$. So $x_1$ is the first value, $x_2$ the second, and so on.
- $\sum_{i=1}^{n}$ ("sigma, from $i=1$ to $n$") — the **summation symbol**. It tells us: *"add up the things to my right, starting with $i = 1$ and going up to $i = n$"*. So $\sum_{i=1}^{n} x_i \;=\; x_1 + x_2 + \cdots + x_n$.
- $\frac{1}{n}$ — divide that sum by $n$.

The formula looks formal, but it is doing exactly what your intuition says: *add them all up, divide by how many*.

#### In the tiny example

$$\bar{x} \;=\; \frac{2 + 4 + 4 + 4 + 5 + 5 + 7 + 9}{8} \;=\; \frac{40}{8} \;=\; 5$$

#### Statistician's vocabulary

When we talk about the entire population (every possible value, not just our sample), the population mean is conventionally written as $\mu$ (Greek *"mu"*). We use $\bar{x}$ for the sample mean and $\mu$ for the population mean. We will return to this distinction in Section 1.8.

### 1.5.2 The median (the middle value)

The **median** is the value that sits in the middle when you sort the numbers from smallest to largest. Half of the values are at or below the median; half are at or above.

#### How it is computed (no single formula)

To find the median:

1. Sort the values from smallest to largest.
2. If $n$ is **odd**, the median is the value at position $\frac{n+1}{2}$.
3. If $n$ is **even**, the median is the **average of the two middle values** — at positions $\frac{n}{2}$ and $\frac{n}{2}+1$.

#### In the tiny example

The values, sorted, are: $2, 4, 4, 4, 5, 5, 7, 9$. There are $n = 8$ values (even), so the median is the average of positions 4 and 5:

$$\text{median} \;=\; \frac{x_{(4)} + x_{(5)}}{2} \;=\; \frac{4 + 5}{2} \;=\; 4.5$$

The notation $x_{(i)}$ (with parentheses around the subscript) means "the $i$-th value **after sorting**" — not the original $i$-th value.

#### Why the median matters

The median is **robust** to outliers: a single huge value pushes the mean up sharply but barely moves the median. In business reporting, the median is often the safer "typical value" to lead with — leadership rarely cares about a number being lifted by one freak hour.

### 1.5.3 The mode (the most common value)

The **mode** is simply the value that appears most often. A dataset can have:

- One mode (called **unimodal**).
- Two modes (called **bimodal**) — equally most-common values.
- Many modes (multimodal).
- No mode if every value is distinct.

#### In the tiny example

The value $4$ appears three times, more than any other value. So the mode is $4$.

The mode is most useful for **categorical** or **count** data. For continuous measurements (like exact weights), it is usually not very meaningful.

### Let's compute mean, median, and mode on our bike data

In [ ]:
# Compute the arithmetic mean of the cnt column (sum of all values divided by how many values).
mean_cnt   = df["cnt"].mean()

# Compute the median of the cnt column (the middle value when sorted).
median_cnt = df["cnt"].median()

# Compute the mode of the cnt column; `.mode()` can return several values, so we keep only the first with `.iloc[0]`.
mode_cnt   = df["cnt"].mode().iloc[0]

# Print the mean.
print(f"Mean rentals per hour:   {mean_cnt:.2f}")

# Print the median.
print(f"Median rentals per hour: {median_cnt:.2f}")

# Print the mode.
print(f"Mode (most common) rentals per hour: {mode_cnt}")

You should see approximately:

- Mean ≈ **189.46**
- Median ≈ **142.00**
- Mode = **5**

Three different numbers, all claiming to be "typical". So which one do we report?

- The **mean** (189.46) is pulled higher by busy commute hours.
- The **median** (142) tells us that **half of all hours have 142 rentals or fewer** — a much safer "typical" for operations planning.
- The **mode** (5) tells us the most common count is just 5 — there are *many* low-traffic hours (nights, off-season, bad weather). That alone is interesting.

> Take a moment. If you were the operations manager at CityCycle and someone said *"on average we rent 189 bikes per hour"*, would that be a useful number for planning? Or misleading?

### 1.5.4 The range, the variance, and the standard deviation — three ways to talk about *spread*

A "typical hour" is only half the story. Two stations could both have the same mean of 189 and behave very differently — one steady around 189, one swinging between 0 and 800. We need to describe **how much the values vary around the mean**.

#### Range

The simplest measure of spread:

$$\text{Range} \;=\; x_{\max} - x_{\min}$$

Where $x_{\max}$ is the largest value and $x_{\min}$ is the smallest. Easy to compute, but very sensitive to extreme single values.

**In the tiny example:** $\text{Range} \;=\; 9 - 2 \;=\; 7$.

#### Variance

The **variance** measures the average **squared distance** of each value from the mean. The formula for the *sample variance* (the version we use when our data is a sample, not the full population) is:

$$s^2 \;=\; \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

Reading every symbol:

- $s^2$ ("s squared") — the sample variance.
- $\bar{x}$ — the sample mean (defined in 1.5.1).
- $x_i - \bar{x}$ — the **deviation** of the $i$-th value from the mean. Positive if the value is above the mean, negative if below.
- $(x_i - \bar{x})^2$ — the squared deviation. Squaring (a) gets rid of the negative signs so positives and negatives don't cancel out, and (b) gives larger weight to bigger deviations.
- $\sum_{i=1}^{n}$ — sum up the squared deviations across all $n$ observations.
- $\frac{1}{n-1}$ — divide by $n-1$ rather than $n$. This is called **Bessel's correction**. Intuitively, when we estimate the mean from the sample we "use up" one degree of freedom; dividing by $n-1$ adjusts for that. (You do not need to memorize this — just know that `pandas` and most software default to $n-1$.)

#### In the tiny example

Mean is 5. Deviations from the mean: $-3, -1, -1, -1, 0, 0, 2, 4$.
Squared: $9, 1, 1, 1, 0, 0, 4, 16$.
Sum of squared deviations: $9 + 1 + 1 + 1 + 0 + 0 + 4 + 16 \;=\; 32$.
Divide by $n - 1 = 7$:

$$s^2 \;=\; \frac{32}{7} \;\approx\; 4.571$$

The number has awkward units (it is in *squared rentals*, *squared dollars*, *squared kilograms* — whatever the original units are, squared). That makes variance hard to interpret directly, which is why we usually take its square root.

#### Standard deviation

The **standard deviation** is the square root of the variance. It is the spread measure you will use 95% of the time in practice.

$$s \;=\; \sqrt{s^2} \;=\; \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Where $s$ is the sample standard deviation. It is in the **same units as the original data** — that is its big advantage. It roughly answers:

> *"On average, how far does a single observation wander from the mean?"*

For the population standard deviation we conventionally use the Greek letter $\sigma$ ("sigma"). We use $s$ for the sample standard deviation and $\sigma$ for the population standard deviation.

#### In the tiny example

$$s \;=\; \sqrt{4.571} \;\approx\; 2.138$$

So our eight numbers wander, on average, about $\pm 2.14$ around their mean of 5. That feels right — most values are within a couple of units of 5.

### Let's compute spread on our bike data

In [ ]:
# Smallest value in the cnt column — the quietest hour in the dataset.
min_cnt   = df["cnt"].min()

# Largest value in the cnt column — the busiest hour in the dataset.
max_cnt   = df["cnt"].max()

# Range = max - min (simplest measure of spread; sensitive to extremes).
range_cnt = max_cnt - min_cnt

# Compute the sample variance of cnt. By default `.var()` uses ddof=1 (i.e., divides by n-1).
var_cnt   = df["cnt"].var()

# Compute the sample standard deviation of cnt — the square root of the variance.
std_cnt   = df["cnt"].std()

# Print the minimum.
print(f"Min:                {min_cnt}")

# Print the maximum.
print(f"Max:                {max_cnt}")

# Print the range.
print(f"Range:              {range_cnt}")

# Print the variance, rounded to 2 decimal places.
print(f"Variance (s^2):     {var_cnt:.2f}")

# Print the standard deviation, rounded to 2 decimal places.
print(f"Standard deviation: {std_cnt:.2f}")

You should see approximately:

- Min = **1**, Max = **977**, Range = **976** — the system has hours where nearly nobody rents, and hours where almost a thousand rentals happen.
- Variance $s^2 \approx$ **32,901.46** — a large number in awkward units (rentals-squared). Mostly an intermediate step toward the standard deviation.
- Standard deviation $s \approx$ **181.39** — a typical hour wanders about $\pm 181$ rentals from the mean of 189. **That is enormous relative to the mean.**

A standard deviation roughly equal to the mean is a strong signal of a very variable, demand-spiked dataset. Operations planning cannot rely on a single number for "expected demand" — context matters a lot (time of day, weather, season).

**A quick consistency check.** If $s \approx 181.39$, then we can verify the variance directly:

$$s^2 \;\approx\; (181.39)^2 \;\approx\; 32{,}902$$

Within rounding, this matches our $s^2 \approx 32{,}901.46$. ✓

### 1.5.5 `describe()` — all summaries at once

`pandas` has a one-call shortcut that gives you the most common summaries at once.

In [ ]:
# `.describe()` is a one-call shortcut: count, mean, std, min, three quartiles (25/50/75%), and max.
df["cnt"].describe()

Reading `.describe()` line by line:

- **count** — how many values there are (17,379, as expected).
- **mean** — the sample mean $\bar{x}$ (≈ 189.46).
- **std** — the sample standard deviation $s$ (≈ 181.39).
- **min** — smallest value (1).
- **25%** — the 25th percentile: 25% of hours are at or below this value (≈ 40).
- **50%** — the 50th percentile, which is the same as the **median** (≈ 142).
- **75%** — the 75th percentile (≈ 281).
- **max** — largest value (977).

The percentile lines tell a richer story than the mean alone. You can already see the distribution is **lopsided**: the jump from 25% (40 rentals) to 75% (281 rentals) is much bigger than the jump from min to 25%, and the mean (189) is well above the median (142). That is the signature of a **right-skewed distribution** — many quiet hours, a long tail of busy ones.

> **Mini-recap.** A typical hour rents about 140–190 bikes, but with enormous swings: from 1 rental at night to 977 at a busy summer rush. The "typical" number depends on which question you ask — mean for "average over all hours", median for "the middle hour". For operations planning, the median is usually the safer number to lead with.

We have not yet *looked* at the data, though. Numbers describe; pictures persuade. On to section 1.6.

***
## 1.6 Visualization — see the data

Numbers are precise but hard to feel. A picture can answer "what does demand look like?" in two seconds, while a paragraph of numbers takes two minutes. In this section we will build the analyst's **starter visualization toolkit**:

1. A **histogram** — to see the overall shape of demand.
2. A **boxplot by hour** — to see the daily commute pattern.
3. A **demand curve** (mean by hour) — same finding, prettier.
4. A **boxplot by weekday** — to see weekday vs. weekend differences.
5. A **boxplot by weather** — to see how weather suppresses demand.

After each plot, we will pause and read what we see.

### Plot 1 — histogram of total rentals

#### What is a histogram?

A **histogram** is a chart that puts each value (or a small range of values) along the horizontal axis and shows how *often* each value occurs by the height of its bar. To build one:

1. Pick a number of equal-width bins (intervals) covering the range of the data.
2. Count how many observations fall into each bin.
3. Draw a bar for each bin, with height equal to the count.

It is the single most useful plot in introductory data analysis — it answers *"what shape is my data?"* at a glance.

In [ ]:
# Create a new matplotlib figure with a specific size (width=10 inches, height=5 inches).
plt.figure(figsize=(10, 5))

# Draw a histogram of the cnt column using 50 bins; `edgecolor="black"` outlines each bar for clarity.
plt.hist(df["cnt"], bins=50, edgecolor="black")

# Set the chart title.
plt.title("Distribution of total bike rentals per hour")

# Label the horizontal axis.
plt.xlabel("rentals per hour (cnt)")

# Label the vertical axis.
plt.ylabel("number of hours")

# Render the plot inside the notebook.
plt.show()

**What we see.** A tall pile on the left (lots of low-rental hours), tapering off to the right (fewer and fewer high-rental hours). This is the **right-skewed** shape we predicted from the percentiles in section 1.5.

In business terms: most hours are quiet; a smaller number of hours are very busy. Capacity planning has to account for the busy tail — staffing for the average will leave you short during peaks.

### Plot 2 — boxplot of rentals by hour of day

#### What is a boxplot?

A **boxplot** is a compact way to show five summary numbers at once:

- The **median** (the line in the middle of the box) — the 50th percentile.
- The **25th and 75th percentiles** (the bottom and top of the box) — together they enclose the middle half of the data.
- The **whiskers** (the lines extending out from the box) — they roughly mark the range of "typical" values; their exact length follows a convention based on the interquartile range.
- The **dots** beyond the whiskers — the **outliers**, unusually high or low values.

A boxplot is the right plot when you want to **compare distributions across groups**. We will plot one box per hour of day (0–23).

In [ ]:
# Create a wide figure — we will draw 24 side-by-side boxes (one per hour of day).
plt.figure(figsize=(12, 5))

# Seaborn boxplot: one box per unique value of `hr` (hour-of-day); cnt values go on the y-axis.
sns.boxplot(data=df, x="hr", y="cnt")

# Set the chart title.
plt.title("Total bike rentals per hour, by hour of day")

# Label the horizontal axis.
plt.xlabel("hour of day (0 = midnight, 12 = noon)")

# Label the vertical axis.
plt.ylabel("rentals in that hour (cnt)")

# Render the plot.
plt.show()

**What we see.** Two unmistakable peaks:

- A morning peak around **hour 8** (commute to work).
- A larger evening peak around **hours 17–18** (commute home).

The middle of the day (10–15) is moderate. The middle of the night (0–5) is essentially empty. This is exactly the demand profile you would expect from a city used heavily by commuters.

### Plot 3 — the demand curve (average rentals by hour)

The boxplot above showed the *distribution* in each hour. Sometimes you just want the **average per hour** as a single line, especially in a memo or slide. That is the **demand curve**.

In [ ]:
# Group rows by hour-of-day, select the cnt column, and compute the mean per hour — returns a small Series.
avg_by_hour = df.groupby("hr")["cnt"].mean()

# Create a wide figure.
plt.figure(figsize=(12, 5))

# Draw a bar chart: x = hour (Series index), y = mean cnt (Series values).
plt.bar(avg_by_hour.index, avg_by_hour.values, edgecolor="black")

# Set the chart title.
plt.title("Average rentals per hour, across the whole 2-year dataset")

# Label the horizontal axis.
plt.xlabel("hour of day")

# Label the vertical axis.
plt.ylabel("mean rentals (cnt)")

# Force one tick per hour on the x-axis (0 through 23).
plt.xticks(range(0, 24))

# Render the plot.
plt.show()

**What we see.** Same story as the boxplot, but as a single clean curve — easier to put on a slide for leadership.

> The line `df.groupby("hr")["cnt"].mean()` is doing a lot of work in one line:
> - `df.groupby("hr")` says: "split the table into groups, one per unique value of `hr`."
> - `["cnt"]` says: "from each group, look at the `cnt` column."
> - `.mean()` says: "compute the mean of `cnt` within each group."
>
> The result is a small table with one row per hour. This `groupby → select column → aggregate` pattern is one of the most common moves in `pandas`.

### Plot 4 — boxplot by day of week

Does weekday-vs-weekend behavior matter? `weekday` is 0 = Sunday, 1 = Monday, …, 6 = Saturday.

In [ ]:
# Create a figure of appropriate size for 7 side-by-side boxes.
plt.figure(figsize=(10, 5))

# Seaborn boxplot: one box per weekday (0..6); cnt values go on the y-axis.
sns.boxplot(data=df, x="weekday", y="cnt")

# Set the chart title.
plt.title("Total bike rentals per hour, by day of week")

# Label the horizontal axis.
plt.xlabel("weekday (0 = Sunday, 6 = Saturday)")

# Label the vertical axis.
plt.ylabel("rentals per hour (cnt)")

# Render the plot.
plt.show()

**What we see.** All seven days look fairly similar at the median, but the *shapes* differ — weekdays (Mon–Fri) have the strong commuting tail (high outliers around the morning and evening peaks), while weekends (Sun, Sat) are flatter. We are seeing the *combination* of a commuting weekday rhythm and a more even leisure-weekend rhythm.

### Plot 5 — boxplot by weather situation

Finally, does weather suppress demand? `weathersit` codes 1 = clear, 2 = misty, 3 = light snow/rain, 4 = heavy weather.

In [ ]:
# Create a figure for 4 side-by-side boxes (one per weather category).
plt.figure(figsize=(10, 5))

# Seaborn boxplot: one box per weathersit code; cnt values on the y-axis.
sns.boxplot(data=df, x="weathersit", y="cnt")

# Set the chart title.
plt.title("Total bike rentals per hour, by weather situation")

# Label the horizontal axis.
plt.xlabel("weather (1=clear, 2=misty, 3=light precip, 4=heavy)")

# Label the vertical axis.
plt.ylabel("rentals per hour (cnt)")

# Render the plot.
plt.show()

**What we see.** Clear weather (1) and misty weather (2) look similar — light mist does not really deter rentals. Light snow/rain (3) drops demand noticeably. Heavy weather (4) is rare in the dataset (very few boxes) but visibly suppresses rentals further.

> **Mini-recap of section 1.6.** Five plots, one story: **demand is driven by time of day and weather**. Two commute peaks dominate weekdays; weekends are flatter; bad weather shaves off rentals. This is the *descriptive* picture of demand. We have answered leadership's first question.

We still have not addressed leadership's *second* question: **how confident are we in any of this?** That is where statistics, and not just numbers, comes in. Sections 1.7–1.10 build the answer.

***
## 1.7 Three "stories" about how data arises

Step back and think about what we just looked at. The histogram of `cnt` was right-skewed: many low-rental hours, fewer and fewer high-rental hours. Where does that shape come from? Why does data look the way it looks?

Statisticians explain "where data comes from" using **probability distributions**. Each distribution is a *story* — a description of the kind of underlying process that produces values like the ones we observed.

The goal for today is simply to **recognize three stories** that show up over and over in business data. We will not do deep computation with each one yet — just learn what each distribution is, how it is parameterized, and what kind of data it describes.

### Where do shapes come from?

Before we meet the three classic distributions, let's slow down and ask a question you might not have thought to ask: **why does data have a shape at all?**

Walk with me through three everyday examples — none of them about bikes.

**Example 1 — Standing on a street corner.** Imagine you stand at a street corner for ten hours one Saturday, holding a clicker, and you count how many cars pass by every minute. By the end of the day you have 600 numbers, one per minute. Some minutes had three cars. Some had twenty. A few had zero. If you plotted those 600 numbers as a histogram, you would see a *shape*: a pile of numbers clustered around some typical value, with fewer minutes at very low or very high counts.

That shape is not an accident. It is the fingerprint of *how cars actually arrive at a corner*. A different street — say, a busy avenue downtown — would have a different shape, but it would still *have* a shape. Shape is what an underlying process leaves behind on the data.

**Example 2 — Lining up by height.** Imagine 500 randomly chosen adults lined up shortest to tallest, with no gaps. The line is *not* uniformly spaced. There is a thick middle of "average-height" people, and the line thins out toward both ends — very short, very tall. The same thing happens with weights, with shoe sizes, with hat sizes, with the daily commute time of a thousand people in your city.

A shape, again. Not random. Reflecting the underlying biology, behaviour, or physics.

**Example 3 — Flipping a coin.** Take a fair coin and flip it 10 times. Count the heads. You might get 3, or 5, or 7. Now do that whole *experiment* 10,000 times — each experiment is a fresh round of 10 coin flips. Plot a histogram of your 10,000 head-counts. You will see a tidy, symmetric shape, centered near 5 (which makes sense — for a fair coin, you expect about half the flips to be heads).

The shape is *built into the rules of the game itself*. You did not choose it; chance did.

**Why does this matter?** Because once you recognize that data has a shape, and that the shape comes from an underlying process, you start to ask different and better questions. Instead of asking *"what value did I get?"* you start to ask *"what kind of process produced this?"*. Once you have an answer to *that* question, you can predict what is likely to happen next, you can plan capacity, you can spot anomalies, and you can reason honestly about uncertainty.

Statisticians have catalogued the most common shapes and given them names. We are about to meet three of those shapes. Each one is a different "story" — a description of one specific kind of underlying process. Recognizing which story fits your data is one of the most valuable skills an analyst can develop, and it does not require any math.

But before we name the three shapes, we need a small piece of vocabulary. That is what the next cell does.

### A foundational definition: what is a probability distribution?

Two terms have to be unpacked before we can sensibly talk about distributions:

- A **random variable** is a quantity whose value depends on chance. Examples: the bike rentals next Monday at 8 a.m. (we do not know yet — that is chance), the number of heads in 10 coin flips, the height of a random person from a population.

  By convention, we use **uppercase letters** for random variables (e.g., $X$, $Y$) and **lowercase letters** for specific values they take (e.g., $x = 5$ means "the value of $X$ is 5").

- A **probability distribution** for a random variable is a rulebook that says **what values that variable can take, and how likely each one is**.

There are two flavors:

- **Discrete** distributions describe variables that can only take specific values, typically counting numbers $0, 1, 2, 3, \ldots$. We describe them with a **probability mass function** (PMF), conventionally written $P(X = k)$ — "the probability that $X$ equals the value $k$".
- **Continuous** distributions describe variables that can take any value in a range. We describe them with a **probability density function** (PDF), conventionally written $f(x)$. The actual probability of an interval is the area under the curve over that interval.

A few rules every distribution obeys:

- Every probability is between $0$ and $1$: $\;0 \;\le\; P(X = k) \;\le\; 1$.
- For a discrete distribution, the probabilities over all possible values sum to $1$: $\;\sum_k P(X = k) \;=\; 1$.
- For a continuous distribution, the total area under the PDF curve equals $1$: $\;\int f(x)\,dx \;=\; 1$.

Now we are ready for the three classic stories.

*A picture before the math — the bell curve is everywhere*

If you have ever heard the phrase *"bell curve"*, you have already met the Normal distribution, even if you did not know the formal name.

Here is a quick way to feel where the bell comes from. Imagine, for a moment, that 100 small influences combine to produce a person's adult height. Maybe 80 genes that each nudge height up or down a little bit, plus another 20 environmental factors (childhood nutrition, sleep, illness, hours of exercise as a child). Each of those 100 influences is *small*. None of them, on its own, decides the person's height. But added together, they *do* decide. And because there are so many of them, and each one is small and somewhat independent of the others, the result tends to **cluster around a typical value**, with fewer and fewer people the further you get from that typical value, in either direction.

This recipe — *"many small independent effects, added together"* — turns out to be so common in nature and in business that the bell shape shows up almost everywhere:

- Adult heights and weights.
- The output of any measurement instrument that has many tiny sources of error.
- The daily revenue of a busy retail store (sums of many independent transactions).
- The total running time of a long manufacturing process with many steps.
- The IQ scores of a large population.
- The errors in repeated rifle shots at a fixed target.
- The annual salaries inside a single job grade in a large company.

In the 19th century, the British scientist **Sir Francis Galton** built a contraption he called the **"bean machine"** (sometimes "Galton board" or *quincunx*). He dropped little beans through a triangular grid of pins. Each pin made the bean bounce left or right by chance — like a tiny, independent coin-flip decision. After dozens of pin bounces, each bean landed somewhere in a row of slots at the bottom. After dropping thousands of beans, the slots filled up in a beautiful pattern: a tall pile in the middle, tapering off symmetrically on both sides. **A bell.**

The Normal distribution is the *precise mathematical description* of that pattern. The next cell introduces it formally — the formula and the symbols — but everything that follows is just a careful way of describing the bell you have already seen in your head.

### Story 1 — The Normal distribution (continuous, symmetric)

The **Normal distribution** is the most famous distribution — the bell-shaped curve. It is the right story for **continuous quantities that arise as the sum or average of many small effects**:

- Adult heights — many genetic + environmental influences add up.
- Manufacturing measurement errors — many tiny sources of error combine.
- Daily revenue totals — sums of many individual transactions.

A Normal distribution is fully described by **two parameters**:

- $\mu$ ("mu") — the mean, which is also the center of the bell.
- $\sigma$ ("sigma") — the standard deviation, which controls the width of the bell.

Its probability density function is:

$$f(x) \;=\; \frac{1}{\sigma \sqrt{2\pi}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Reading every symbol:

- $f(x)$ — the height of the density curve at the value $x$ (how concentrated the probability is around that value).
- $\mu$ — the mean (center of the bell).
- $\sigma$ — the standard deviation (width of the bell). $\sigma^2$ is the variance.
- $\pi \;\approx\; 3.1416$ — the familiar geometric constant.
- $\exp(z) \;=\; e^z$ — the exponential function. $e \;\approx\; 2.718$ is Euler's number.

**You do not need to memorize this formula.** Two things matter:

1. The peak is at $x = \mu$ (the mean equals the median equals the mode for a Normal).
2. About 68% of the probability falls within $\mu \pm \sigma$, about 95% within $\mu \pm 2\sigma$, and about 99.7% within $\mu \pm 3\sigma$. This is the **68–95–99.7 rule**.

#### Tiny example

Adult male heights in some country might be Normal with $\mu = 175$ cm and $\sigma = 7$ cm. We write:

$$X \;\sim\; N(\mu, \sigma^2) \;=\; N(175,\ 49)$$

The notation $X \sim N(\mu, \sigma^2)$ reads *"$X$ is distributed as Normal with mean $\mu$ and variance $\sigma^2$"*. The squiggly $\sim$ ("tilde") is read *"is distributed as"*.

Applying the 68–95–99.7 rule: roughly 68% of men in this population are between 168 cm and 182 cm; 95% are between 161 cm and 189 cm; 99.7% are between 154 cm and 196 cm.

*A picture before the math — counting yes-or-no*

Now a different kind of question. Instead of asking *"how big is the value?"*, we ask *"how many yes-es did we get?"*

Whenever you have a **fixed number of independent trials**, each of which has only **two possible outcomes** — *yes* or *no*, *heads* or *tails*, *clicked* or *did not click*, *converted* or *bounced*, *passed* or *failed* — the count of "yes-es" follows the Binomial story.

Some examples to make this concrete:

- You flip a fair coin 10 times and write down how many came up heads. The count could be anything from 0 to 10, but values near 5 are far more likely than values near 0 or 10.
- You send 100 marketing emails. Each recipient either opens it or does not. You count the openings. Some campaigns land 20 opens, some 30, some 45 — but again, the count clusters around a typical value that depends on how appealing your subject line is.
- A pollster calls 500 people and counts how many say *yes* to a specific question. The count varies from poll to poll, even if nothing in the underlying population has changed.
- A factory ships 1,000 widgets. Each one either passes quality control or fails. The count of failures is Binomial.
- You drive past a row of 20 traffic lights on your commute. The count of green ones you pass through is roughly Binomial.

Two numbers fully define a Binomial story:

1. **How many trials you ran.** Call it $n$.
2. **How likely each individual trial was to come out a "yes".** Call it $p$.

That is everything. Once $n$ and $p$ are pinned down, the entire shape — the entire probability distribution — is determined. There is no extra freedom; no other parameters to choose. This is rare and beautiful.

Here is something charming. When $n$ is small (say, 10), the Binomial shape looks "chunky" — only a small set of counts is possible (0 through 10), and the histogram has clearly separate bars. But as $n$ grows large (say, 200 or 1,000 or 10,000), the Binomial shape **starts to look more and more like the bell curve we just discussed.** That is not a coincidence — it is a quiet preview of the Central Limit Theorem, which is coming up in §1.9.

The next cell gives the formal definition, with the formula.

### Story 2 — The Binomial distribution (counts of yes/no successes)

The **Binomial distribution** is the right story for **counts of "yes/no" outcomes in a fixed number of trials**:

- Out of 200 visitors to a webpage, how many converted?
- Out of 50 coin flips, how many came up heads?
- Out of 100 commuters arriving at a CityCycle station at 8 a.m., how many actually rented a bike?

A Binomial distribution has **two parameters**:

- $n$ — the number of trials (a positive whole number).
- $p$ — the probability of "success" on a single trial ($0 \le p \le 1$).

Its probability mass function is:

$$P(X = k) \;=\; \binom{n}{k}\, p^k\, (1-p)^{n-k}$$

Reading every symbol:

- $X$ — the random variable: the number of successes in $n$ trials.
- $k$ — the specific count we are asking about ($k$ can be $0, 1, 2, \ldots, n$).
- $p^k$ — the probability of getting $k$ successes (multiplying success-probability $k$ times).
- $(1-p)^{n-k}$ — the probability of getting $n-k$ failures (multiplying failure-probability $n-k$ times).
- $\binom{n}{k}$ — read *"n choose k"*, the number of distinct ways to pick which $k$ of the $n$ trials are the successes. Formally: $\binom{n}{k} \;=\; \frac{n!}{k!(n-k)!}$.
- $n! \;=\; n \times (n-1) \times \cdots \times 2 \times 1$ — read *"n factorial"*.

We write $X \sim \text{Binomial}(n, p)$ to mean *"$X$ follows a Binomial distribution with $n$ trials and success probability $p$"*.

#### Tiny example

Flip a fair coin 10 times and count the heads. Then $X \sim \text{Binomial}(10, 0.5)$. The probability of getting exactly 5 heads is:

$$P(X = 5) \;=\; \binom{10}{5}\, (0.5)^5\, (0.5)^{5} \;=\; 252 \times \frac{1}{1024} \;\approx\; 0.246$$

So about a 24.6% chance — the most likely single outcome, but far from a guarantee.

*A picture before the math — counting things that just happen*

Now a third kind of counting story — and this one will feel familiar the moment you spot it.

Imagine you run a quiet neighborhood cafe. You are curious about your customer flow, so one day you sit at the counter with a notebook and count how many customers walk in during each 15-minute window. Over the course of a whole day you have a list of numbers: $2, 5, 1, 0, 3, 4, 2, 7, 0, 1, 2, \ldots$. Each number is the count of customers in one 15-minute window.

Notice what is different from the Binomial. There is **no "n trials"** here. There is no upfront pool of "potential customers" who each independently decided yes-or-no. Customers just arrive when they arrive. Some windows happen to be quiet; some happen to be busy. The events are *not* triggered by a fixed number of yes/no choices — they are *triggered by the passage of time itself*.

This is the **Poisson story**. It describes counts of events happening in a *fixed window of time or space*, when those events occur somewhat independently and at some steady average rate.

You meet Poisson counts constantly in business and in science:

- Customers walking into a quiet shop, per 15-minute window.
- Phone calls reaching a help desk, per minute.
- Emails arriving in your inbox, per hour.
- Buses passing a particular stop, per hour.
- Typos per page in a 300-page book draft.
- Earthquakes above magnitude 6, per year, worldwide.
- Goals scored by one team, per soccer match.
- **Bike rentals per hour at a quiet CityCycle station.**

The single number that controls a Poisson distribution is the **average rate** of events per window, traditionally called $\lambda$ (the Greek letter *lambda*). A cafe with $\lambda = 4$ customers per 15-minute window will see lots of windows near 4, plenty with 2 or 3, occasional windows with 0 or 7 or 8 — and almost never 50.

A nice property to remember: for the Poisson distribution, the **mean** and the **variance** of the count are both equal to $\lambda$. The bigger the average rate, the wider the spread — and the more bell-shaped the histogram becomes. (Yet another quiet preview of the Central Limit Theorem.)

Compare this to the Binomial we just met:

- **Binomial counts** answer: *"how many yes-es out of $n$ tries?"*
- **Poisson counts** answer: *"how many events happened in this window?"*

Different stories. Used for different kinds of business measurements. Recognizing which one fits your data is more useful than memorizing any formula.

The next cell gives the formal definition, with the formula.

### Story 3 — The Poisson distribution (counts of events per window)

The **Poisson distribution** is the right story for **counts of (relatively rare) events happening in a fixed window of time or space**:

- Phone calls arriving at a help desk per minute.
- Goals scored in a soccer match.
- Bike rentals per hour at a quiet station.

A Poisson distribution has **one parameter**:

- $\lambda$ ("lambda") — the *expected* (average) number of events per window. $\lambda > 0$.

Its probability mass function is:

$$P(X = k) \;=\; \frac{\lambda^{k}\, e^{-\lambda}}{k!}$$

Reading every symbol:

- $X$ — the random variable: the count of events in one window.
- $k$ — the specific count we are asking about ($k$ can be $0, 1, 2, 3, \ldots$). Note that $k$ has no upper bound in principle.
- $\lambda^k$ — the parameter $\lambda$ raised to the $k$-th power.
- $e \;\approx\; 2.718$ — Euler's number.
- $e^{-\lambda}$ — Euler's number to the negative-$\lambda$ power; this is a small number that ensures the total probability sums to 1.
- $k!$ — $k$ factorial (see Binomial above).

We write $X \sim \text{Poisson}(\lambda)$.

A nice property: the mean of a Poisson is $\lambda$, and the variance is also $\lambda$. So a Poisson with $\lambda = 8$ has both mean 8 and variance 8.

#### Tiny example

Suppose a quiet CityCycle station gets an average of $\lambda = 3$ rentals per hour. Then the probability of zero rentals in a given hour is:

$$P(X = 0) \;=\; \frac{3^{0}\, e^{-3}}{0!} \;=\; \frac{1 \cdot e^{-3}}{1} \;=\; e^{-3} \;\approx\; 0.0498$$

So about 5% of hours at that station are completely empty — a useful number for operations planning.

### Let's simulate all three side by side

`numpy` gives us functions to draw random values from each story: `np.random.normal`, `np.random.binomial`, `np.random.poisson`. We will draw 10,000 values from each and plot the histograms.

In [ ]:
# Seed numpy's random generator so the random draws below are reproducible (same numbers every time).
np.random.seed(42)

# Draw 10,000 values from a Normal distribution with mean=170 and std=10.
normal_samples   = np.random.normal(loc=170, scale=10, size=10000)

# Draw 10,000 values from a Binomial distribution with n=200 trials and p=0.12 success probability.
binomial_samples = np.random.binomial(n=200, p=0.12, size=10000)

# Draw 10,000 values from a Poisson distribution with rate parameter lambda=8.
poisson_samples  = np.random.poisson(lam=8, size=10000)

# Create a figure with 3 side-by-side subplots; `fig` is the whole figure, `axes` is the array of three subplots.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Draw the Normal histogram in the first (leftmost) subplot.
axes[0].hist(normal_samples, bins=40, edgecolor="black")

# Title for the first subplot (uses LaTeX between dollar signs).
axes[0].set_title("Normal: continuous, symmetric\n$X\\sim N(170, 10^2)$")

# Label the x-axis of the first subplot.
axes[0].set_xlabel("value")

# Label the y-axis of the first subplot.
axes[0].set_ylabel("frequency")

# Draw the Binomial histogram in the middle subplot.
axes[1].hist(binomial_samples, bins=40, edgecolor="black")

# Title for the middle subplot.
axes[1].set_title("Binomial: count of successes\n$X\\sim \\mathrm{Binomial}(200, 0.12)$")

# Label the x-axis of the middle subplot.
axes[1].set_xlabel("value")

# Label the y-axis of the middle subplot.
axes[1].set_ylabel("frequency")

# Draw the Poisson histogram in the third (rightmost) subplot.
axes[2].hist(poisson_samples, bins=40, edgecolor="black")

# Title for the third subplot.
axes[2].set_title("Poisson: rare events per window\n$X\\sim \\mathrm{Poisson}(8)$")

# Label the x-axis of the third subplot.
axes[2].set_xlabel("value")

# Label the y-axis of the third subplot.
axes[2].set_ylabel("frequency")

# Adjust subplot spacing so titles and labels do not overlap each other.
plt.tight_layout()

# Render the full figure.
plt.show()

**What we see.**

- The **Normal** histogram is a clean bell shape around 170, with most values between 160 and 180.
- The **Binomial** histogram looks bell-shaped too (because $n$ is large) — centered near $n \cdot p = 200 \times 0.12 = 24$.
- The **Poisson** histogram is right-skewed, centered near $\lambda = 8$, with a hard floor at zero.

### Which story fits our bike rentals?

Compare the Poisson plot above to our `cnt` histogram from section 1.6 — same right-skewed shape, with a floor at zero and a long tail to the right. Bike rentals per hour at a *quiet* station are very close to a Poisson story.

At a *busy* station or hour, the count gets so large that the distribution starts to look more Normal. This is not a coincidence — it is a preview of the **Central Limit Theorem**, which we will meet in section 1.9.

> **Mini-recap.** We now have a vocabulary for the *shapes* of data: Normal (symmetric, continuous), Binomial (yes/no counts), Poisson (rare-event counts). Each one is described by a formula with named parameters that we have read symbol by symbol. Recognizing which story fits a business measurement is more valuable than memorizing any formula — it tells you which tools will and will not work.

Ask your tutor: *"Why do counts of rare events follow a Poisson distribution? Can you give me a small intuition?"*

*A pause before the next section.*

We have spent §1.5 through §1.7 *looking at our data*. We computed its center (mean, median, mode), its spread (variance, standard deviation), drew pictures of its shape, and put names to that shape (Poisson-ish in the quiet hours, drifting toward Normal in the busy ones). That is a lot of progress on leadership's first question: *"what does demand actually look like?"*

But we have been quietly skipping over a question that should bother us: **how do we know any of these numbers are right?**

We computed a mean of 189.46 rentals per hour. Should we put that number on a slide and walk into a meeting? What if our two years of data had come out slightly differently — would the mean have come out slightly differently too? *How much* differently?

This is leadership's second question, and the rest of the notebook is dedicated to answering it. We will not give a fuzzy answer like *"the mean is approximately 189"*. We will give a *quantified* answer — we will tell leadership how much trust to put in the number 189, expressed as a number itself.

The rest of this week's analysis lives inside this question. And to answer it, we need to slow down and think carefully about something we have so far taken for granted: the difference between **the data we have** and **the data we could have**.

***
## 1.8 Sample versus population

Now we pivot to leadership's *second* question: **how confident are we in any of this?**

Look at the mean of `cnt` again: about $189.46$. Where did that number come from? From the 17,379 hours we happen to have in this file. But CityCycle did not stop running on December 31, 2012 — the bike-sharing system kept producing more hours of data, and similar systems in other cities produce data too.

In statistics we draw a careful distinction:

- The **population** is the idealized *full set* of values you could in principle observe. For CityCycle, you can think of the population as "all hours of bike-rental demand under operating conditions like ours" — a lot of hours, including ones we will never directly observe. Population quantities are typically written with **Greek letters**: the population mean is $\mu$ ("mu"); the population standard deviation is $\sigma$ ("sigma").
- The **sample** is what we actually have in hand. Our 17,379 hours are a sample drawn from that imagined population. Sample quantities are typically written with **Latin letters with a bar or hat**: the sample mean is $\bar{x}$, the sample standard deviation is $s$.

This matters because **the mean we computed is a sample mean ($\bar{x}$), not the population mean ($\mu$)**. The two will be close — if our sample is large and reasonably representative — but they are not the same number.

### A tiny demo of sampling variability

Let's pretend, just for the next two sections, that the 17,379 hours in `df` *are* the population. We will then draw small samples from that "population" and see how the sample mean wobbles around the true population mean $\mu$.

In [ ]:
# Extract the cnt column as a plain numpy array; we will treat this array as the "population".
population = df["cnt"].values

# Compute the true population mean (mu) by taking the mean of every value in the population.
true_pop_mean = population.mean()

# Print the population mean, formatted to two decimal places.
print(f"True population mean (mu): {true_pop_mean:.2f}")

# Print a blank line for readability.
print()

# Seed numpy's random generator so the samples below are reproducible.
np.random.seed(0)

# Print a heading before the per-sample output.
print("Five sample means (each from a fresh sample of 50 hours):")

# Loop five times, drawing a fresh sample of 50 hours each time, and printing its mean.
for i in range(5):
    # Draw 50 random values from the "population", with replacement (each value can be picked more than once).
    sample = np.random.choice(population, size=50, replace=True)
    # Print the index of this sample (1-indexed) and the sample mean, formatted to 2 decimals.
    print(f"  Sample {i + 1}: x-bar = {sample.mean():.2f}")

**What we see.** Five samples drawn from the *same* population give five *different* sample means $\bar{x}_1, \bar{x}_2, \ldots, \bar{x}_5$. They are roughly in the right neighborhood, but each one wobbles around the true population mean $\mu \approx 189.46$.

This is **sampling variability**. It is not a bug in the data; it is a fundamental property of using samples. If we shipped the result from any single one of those five samples to leadership without acknowledging this, we would be implicitly pretending we know the answer more precisely than we do.

> **Mini-recap.** A single mean is one realization of a process that has variability built into it. The population $\mu$ is what we wish we knew; the sample $\bar{x}$ is what we can compute. They differ. To honestly say "how confident we are" we need to *quantify* the wobble. That is section 1.9.

***
## Before §1.9 — the most beautiful theorem in statistics

We are about to meet the single most important idea in introductory statistics — and arguably the most beautiful theorem ever proved in the field. Settle in. We will take our time with this one, because if it lands properly, an enormous amount of the rest of statistics suddenly feels obvious.

### A thought experiment

Imagine, for a moment, **a thousand parallel universes**. In each one, a junior analyst at a different city's bike-share company is sitting at a desk this Friday afternoon, doing exactly the same job you are doing. Each one has their own dataset, drawn from operating conditions roughly like CityCycle's. Each one has just computed the mean hourly demand from their data.

You phone all 1,000 of them and ask: *"What number did you get?"* You write down all 1,000 answers on a long sheet of paper. Then you draw a histogram of those 1,000 numbers.

**What shape does that histogram have?**

Take a moment with the question. It is not obvious. The raw bike-rental data — the `cnt` column we have been working with — was *not* bell-shaped. It was strongly right-skewed, with a thick pile of quiet hours and a long tail of busy ones. You might therefore guess that the 1,000 sample means from the parallel universes would *also* be right-skewed, since they came from right-skewed data.

The answer — and it genuinely feels like a small miracle the first time you see it — is:

> **The 1,000 sample means form a beautiful, symmetric, bell-shaped distribution. Even though the raw data is not bell-shaped at all.**

That is the **Central Limit Theorem**.

### A second way to see it: the wisdom of crowds for chance

Here is another angle, from a totally different setting.

Imagine a single archer. A talented one — but still human. She fires a single arrow at a target. Where does it land? Probably somewhere on the target, but not exactly in the middle. There is some random wobble — wind, breath, fatigue, the angle her elbow happens to be at.

Now imagine she fires *thirty* arrows in a row, and at the end she calculates the **average position** of where the thirty arrows landed. That average position will sit *much closer to the center of the target* than any single individual arrow. Random wobbles in either direction tend to **cancel each other out** when you take an average.

Now imagine she repeats this entire thirty-arrow exercise on a hundred different days. Each day she gets one average position. You plot those 100 averages on the target.

The 100 averages cluster *tightly* around the bullseye — much more tightly than any single arrow ever lands. And the distribution of those 100 averages — even though each individual arrow is wobbly — is **bell-shaped**.

That is also the Central Limit Theorem.

### Why this matters for your CityCycle job

In a moment we will run the parallel-universes thought experiment for real, using a computer simulation. We will pretend our 17,379 hours of data *are* "the population", draw 10,000 fresh samples from it, compute the mean of each one, and look at the distribution of those 10,000 means.

The result will be a bell curve, exactly as the theorem predicts.

**That bell curve is what allows you, as an analyst, to say something honest about how trustworthy your reported mean is.** If the bell is wide, the answer is *"not very"*. If the bell is narrow, the answer is *"very trustworthy"*. The *width* of that bell is the number that turns a single computed average into a defensible business report.

We are about to make that bell appear with about ten lines of Python. But before we do, the next cell states the theorem precisely — with the formula and the symbols — so the simulation has a name to attach itself to.

***
## 1.9 The sampling distribution of the mean (Central Limit Theorem)

In section 1.8 we drew five samples and got five different sample means. Now we will do the same thing **ten thousand times** and look at the *distribution of those sample means*. That distribution has a name: the **sampling distribution of the mean**.

### A new concept: the sampling distribution

For a fixed sample size $n$, the **sampling distribution of the mean** is the probability distribution of the random variable $\bar{X}_n$ — *"the mean you would get if you drew a fresh sample of size $n$"*. Drawing a single sample and computing its mean gives you one *realization* of $\bar{X}_n$; doing it 10,000 times gives us a *picture* of its distribution.

This is one of the most important ideas in introductory statistics. Read it carefully.

### The Central Limit Theorem (CLT) — stated formally

**The Central Limit Theorem says:**

> *Let $X_1, X_2, \ldots, X_n$ be a sample of $n$ independent observations drawn from any distribution with mean $\mu$ and finite variance $\sigma^2$. Let $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$ be the sample mean. Then as $n$ grows large, the distribution of $\bar{X}_n$ becomes approximately Normal:*

$$\bar{X}_n \;\sim\; N\!\left(\mu,\ \frac{\sigma^2}{n}\right) \quad \text{(approximately, for large } n\text{)}$$

Reading every symbol:

- $X_1, X_2, \ldots, X_n$ — the $n$ individual observations in our sample (think: 50 random hours of bike rentals).
- $\mu$ — the population mean (the true, idealized average).
- $\sigma^2$ — the population variance.
- $\bar{X}_n$ — the **random variable** "sample mean from a sample of size $n$".
- $N\!\left(\mu,\ \sigma^2/n\right)$ — a Normal distribution with mean $\mu$ and variance $\sigma^2/n$.
- $\sim$ ("is distributed as") — the symbol that says "this quantity has this distribution".

The CLT does **two miraculous things**:

1. **It makes the sample mean Normal**, even when the raw data is not. (Bike rentals are right-skewed; sample means of bike rentals are bell-shaped.)
2. **It shrinks the variance of the sample mean by a factor of $n$.** Specifically, $\operatorname{Var}(\bar{X}_n) = \sigma^2 / n$. That is the math behind why bigger samples give more trustworthy means.

### The simulation

We will:

1. Draw a fresh random sample of size $n = 50$ from the population.
2. Compute that sample's mean $\bar{x}$.
3. Remember it.
4. Repeat steps 1–3 a total of 10,000 times.
5. Plot a histogram of all 10,000 sample means.

That histogram *is* an empirical picture of the sampling distribution of $\bar{X}_{50}$.

In [ ]:
# Seed numpy's random generator so the simulation below is reproducible.
np.random.seed(42)

# Size of each individual sample we draw from the population.
n = 50

# Number of samples to draw — each one will give us one sample mean.
num_samples = 10000

# Build a numpy array of 10,000 sample means: draw a sample of size n, compute its mean, repeat 10,000 times.
sample_means_50 = np.array([
    np.random.choice(population, size=n, replace=True).mean()
    for _ in range(num_samples)
])

# Create a figure of fixed size for the histogram.
plt.figure(figsize=(10, 5))

# Draw a histogram of the 10,000 sample means using 50 bins.
plt.hist(sample_means_50, bins=50, edgecolor="black")

# Draw a vertical red dashed line at the true population mean, for visual comparison.
plt.axvline(true_pop_mean, color="red", linestyle="--", label=f"True population mean μ = {true_pop_mean:.1f}")

# Set the chart title.
plt.title(f"Distribution of {num_samples} sample means (sample size n = {n})")

# Label the horizontal axis.
plt.xlabel("sample mean")

# Label the vertical axis.
plt.ylabel("number of samples")

# Show the legend (so the red dashed line is labelled).
plt.legend()

# Render the plot.
plt.show()

**What we see.**

The shape is a clean, symmetric **bell** centered very close to the true population mean ($\mu \approx 189.46$) — even though the raw `cnt` data is *not* bell-shaped (recall: it was right-skewed, with most hours having low counts). This is the CLT in action.

### The role of sample size

Now let's repeat the simulation with **larger samples** — $n = 200$ instead of $n = 50$ — and see what changes.

In [ ]:
# Seed numpy's random generator so the second simulation is reproducible too.
np.random.seed(42)

# Larger sample size for the second simulation.
n_large = 200

# Build a numpy array of 10,000 sample means, this time with samples of size n_large.
sample_means_200 = np.array([
    np.random.choice(population, size=n_large, replace=True).mean()
    for _ in range(num_samples)
])

# Create a figure with 2 side-by-side subplots that share the same x-axis range, for fair visual comparison.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

# Left subplot: histogram of sample means for n = 50.
axes[0].hist(sample_means_50, bins=50, edgecolor="black")

# Left subplot: red dashed reference line at the true population mean.
axes[0].axvline(true_pop_mean, color="red", linestyle="--", label=f"μ = {true_pop_mean:.1f}")

# Left subplot title — includes the empirical spread (standard deviation) of the 10,000 sample means.
axes[0].set_title(f"n = 50  (spread = {sample_means_50.std():.2f})")

# Left subplot x-axis label.
axes[0].set_xlabel("sample mean")

# Left subplot y-axis label.
axes[0].set_ylabel("number of samples")

# Show the legend on the left subplot.
axes[0].legend()

# Right subplot: histogram of sample means for n = 200.
axes[1].hist(sample_means_200, bins=50, edgecolor="black")

# Right subplot: red dashed reference line at the true population mean.
axes[1].axvline(true_pop_mean, color="red", linestyle="--", label=f"μ = {true_pop_mean:.1f}")

# Right subplot title — includes the empirical spread of the 10,000 sample means (should be smaller).
axes[1].set_title(f"n = 200 (spread = {sample_means_200.std():.2f})")

# Right subplot x-axis label.
axes[1].set_xlabel("sample mean")

# Right subplot y-axis label.
axes[1].set_ylabel("number of samples")

# Show the legend on the right subplot.
axes[1].legend()

# Adjust spacing so the subplot titles and labels do not overlap.
plt.tight_layout()

# Render the figure.
plt.show()

**What we see.** Both histograms are bell-shaped and centered on the same true mean $\mu$. **The right-hand one is narrower.** Larger samples produce sample means that vary less, exactly as the CLT predicts (the variance of $\bar{X}_n$ is $\sigma^2 / n$, so quadrupling $n$ halves the spread).

In plain English: *the more data your average is built on, the more you can trust it.* This is the single most important honest-uncertainty insight in introductory statistics.

> **Mini-recap.** The mean of *one* sample is one number. The means of *many* samples form a distribution. That distribution is bell-shaped and centered on the truth $\mu$ (CLT). Its width shrinks as your sample size grows. We can now put a *number* on "how confident we are" — section 1.10.

***
## Before §1.10 — two kinds of "wobble"

Take a breath. We just watched the Central Limit Theorem appear in front of us. The distribution of 10,000 sample means came out bell-shaped, exactly as the theorem promised, and the bell got narrower when we increased the sample size from 50 to 200.

Now we need to make one more careful distinction — one that, when missed, leads to embarrassing mistakes in business reports.

### Two completely different things you could mean by "wobble"

Look back at our `cnt` data. There are two completely different kinds of wobble or variation you could talk about, and they are **not the same number**.

**Wobble #1 — How much does a single hour vary from the average?**

Some hours, only 5 bikes are rented. Other hours, 800. There is enormous variation between individual hours. We already have a number for this kind of wobble — it is the **standard deviation** of the raw data, $s \approx 181$. That number says, in plain English: *"on average, a single hour wanders about 181 rentals away from the overall mean of 189."*

**Wobble #2 — How much does my COMPUTED AVERAGE wobble if I redo the calculation on a different sample?**

This is a totally different question, and we just saw, in §1.9, that the answer is *much* smaller than wobble #1. In our simulation, the spread of the 10,000 sample means (with $n = 50$) was only about 26. With $n = 200$, it was only about 13. With $n = 17{,}379$ — our whole dataset — it is, as we will compute in a moment, only about 1.4.

So even though individual hours vary by $\pm 181$ rentals, the *average* of 17,379 hours only varies by about $\pm 1.4$ if we kept redoing the experiment in parallel universes. That is an astonishing reduction.

These two wobbles answer **two different questions**:

- **Wobble #1** tells you *how variable the underlying business is*. It is a property of CityCycle's demand pattern.
- **Wobble #2** tells you *how trustworthy the number you are reporting is*. It is a property of your analysis, not of CityCycle's demand.

You need both — but you need to **report them differently**, because they mean different things to leadership.

### An archery analogy

Back to the archer.

- The **standard deviation** is *how scattered a single archer's individual arrows are*. It is a property of the archer (or of the conditions she is shooting in).
- The **standard error** is *how much the AVERAGE of her thirty-arrow rounds shifts if you ran the round again on a different day*. It depends on her individual scatter, *and* on how many arrows you are averaging over.

A more skilled archer has a smaller standard deviation. A larger sample size (more arrows per round) shrinks the standard error.

They are different numbers measuring different things. People in business reports constantly confuse them — they will report an average with the *"standard deviation"* attached, when what they should be reporting is the *standard error*. The next cell makes the distinction precise, with a formula, and then verifies it on the bike-rental data.

***
## 1.10 Standard deviation versus standard error

We have used the word **standard deviation (SD)** several times. There is a related but distinct quantity called the **standard error (SE)** that comes from the sampling distribution we just simulated. Confusing the two is one of the most common mistakes in business reporting.

### The two quantities side by side

| Quantity | Describes the spread of … | Symbol | Practical question it answers |
|---|---|---|---|
| **Standard deviation** | the *raw observations* (individual hours) | $\sigma$ (population) or $s$ (sample) | "How variable is a single hour's demand?" |
| **Standard error** | an *estimate* (a sample mean) | SE or $\sigma_{\bar{X}}$ | "How much should I trust the mean I just reported?" |

### The relationship — from the CLT

The CLT tells us that $\operatorname{Var}(\bar{X}_n) = \sigma^2 / n$. Taking the square root gives the **standard error of the sample mean**:

$$\operatorname{SE}(\bar{X}_n) \;=\; \sqrt{\operatorname{Var}(\bar{X}_n)} \;=\; \frac{\sigma}{\sqrt{n}}$$

Reading every symbol:

- $\operatorname{Var}(\bar{X}_n)$ — the variance of the sample-mean random variable.
- $\operatorname{SE}(\bar{X}_n)$ — the standard error of the sample mean. It is itself a standard deviation — specifically, the standard deviation of the sampling distribution of $\bar{X}_n$.
- $\sigma$ — the population standard deviation.
- $n$ — the sample size.
- $\sqrt{n}$ — the square root of $n$.

In practice we don't know $\sigma$ exactly — we estimate it with the sample standard deviation $s$:

$$\widehat{\operatorname{SE}} \;=\; \frac{s}{\sqrt{n}}$$

The hat ($\widehat{\,\,}$) means *"estimated from data"*.

Two consequences pop out of this formula:

1. **SE shrinks as $n$ grows** — because $\sqrt{n}$ is in the denominator. To halve the SE you need **four times** as many observations (since $\sqrt{4} = 2$).
2. **SE is always smaller than $s$** for any sample of size greater than 1 (since $\sqrt{n} > 1$).

Let's verify the formula by comparing two ways to compute SE.

In [ ]:
# The sample standard deviation of the raw cnt column (this is what pandas .std() gives us, with ddof=1).
sd_raw = df["cnt"].std()

# Empirical SE for n=50: the standard deviation of the 10,000 sample means we simulated earlier.
se_empirical_50  = sample_means_50.std()

# Empirical SE for n=200: same idea, using the second simulation.
se_empirical_200 = sample_means_200.std()

# Formula-based SE for n=50: divide the raw sample SD by sqrt(50).
se_formula_50  = sd_raw / np.sqrt(50)

# Formula-based SE for n=200: divide the raw sample SD by sqrt(200).
se_formula_200 = sd_raw / np.sqrt(200)

# Print the raw-data standard deviation.
print(f"SD of raw cnt (s):                              {sd_raw:.2f}")

# Print a blank line for readability.
print()

# Print the empirical SE at n=50.
print(f"SE for n=50  via simulation:                    {se_empirical_50:.2f}")

# Print the formula-based SE at n=50 — should match the empirical one within a hair.
print(f"SE for n=50  via formula  (s / sqrt(50)):       {se_formula_50:.2f}")

# Print a blank line for readability.
print()

# Print the empirical SE at n=200.
print(f"SE for n=200 via simulation:                    {se_empirical_200:.2f}")

# Print the formula-based SE at n=200 — should also match within a hair.
print(f"SE for n=200 via formula  (s / sqrt(200)):      {se_formula_200:.2f}")

**What we see.** The empirical SE (from the 10,000 simulations) and the formula-based SE match within a hair. The formula works.

Practical translations of the numbers:

- **SD ≈ 181.39**: a typical single hour wanders about $\pm 181$ rentals from the overall mean. Demand is genuinely variable.
- **SE for $n = 50$ ≈ 25.65**: if you computed the mean from a random sample of 50 hours, that mean would typically wander about $\pm 26$ rentals from the true mean.
- **SE for $n = 200$ ≈ 12.83**: with 200 hours, the typical wobble shrinks to about $\pm 13$. Halving SE required four times the data, exactly as the $\sqrt{n}$ in the formula predicts.

### So how trustworthy is *our* reported mean?

We have $n = 17{,}379$ hours of data. The standard error of our overall mean is:

$$\widehat{\operatorname{SE}} \;=\; \frac{s}{\sqrt{n}} \;=\; \frac{181.39}{\sqrt{17{,}379}} \;\approx\; 1.38$$

So our reported mean of 189.46 has a typical wobble of only about $\pm 1.4$. The number is trustworthy — much more trustworthy than the hour-to-hour variation in demand itself, which has SD ≈ 181.

This is the answer to leadership's second question. We can finally say honestly:

> *"Our reported hourly mean is built on 17,379 hours of data, so the wobble in the reported mean is very small — much less than the hour-to-hour variation in demand itself. The number we report as the average is trustworthy; planning on the average alone is not, because the **spread** is large."*

> **Mini-recap.** SD tells you the spread of *single observations*. SE tells you the spread of an *estimate*. SD stays put as data grows; SE shrinks. Reporting "our mean is $\bar{x}$ with $\operatorname{SE} = \widehat{\operatorname{SE}}$" is far more honest than reporting "our mean is $\bar{x}$" alone.

Ask your tutor: *"Why does SE shrink with sample size, but SD does not?"*

***
## 1.11 Our first Claude API call from Python

We have computed a lot. To finish the week and deliver a memo, we need to **translate the numbers into words leadership can read**. We will use two tools side by side:

- **Python** — to compute (which is what we have done so far).
- **Claude** — to help phrase the findings for a non-technical reader.

This is the very first time in the course we will let our code talk to Claude. We will keep it small and explicit. Later sessions will introduce more structured patterns (JSON schemas, multi-turn conversations, etc.).

### Step 1 — confirm the API key is available

In the repository's main `README.md` you set the environment variable `ANTHROPIC_API_KEY` to your personal API key, and restarted VS Code so that Python can see it. The cell below confirms the key is visible. **If it is missing, this cell stops the notebook with a clear message** — do not try to "fix" it in code. Go back to the README, follow Step 4 ("Store the API key on your Windows machine"), close VS Code completely, reopen it, and re-run from this cell.

In [ ]:
# Import Python's `os` module so we can read operating-system environment variables.
import os

# Try to read the ANTHROPIC_API_KEY from the environment; returns None if it is not set.
api_key = os.environ.get("ANTHROPIC_API_KEY")

# If the key is not set, stop the notebook with a clear message (do not continue with a missing key).
if not api_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Open the repository README (top-level), follow Step 4 ('Store the API key on your Windows machine'),\n"
        "close VS Code completely, reopen it, and re-run this cell."
    )

# Confirm the key is set without printing the key itself (only its character length is safe to log).
print(f"ANTHROPIC_API_KEY is set. Key length: {len(api_key)} characters.")

### Step 2 — create the Claude client

The `anthropic` library (which we installed in Session 00) provides a `Client` object that handles all the network plumbing for us. We never have to think about HTTP, headers, or authentication tokens. The client object will read `ANTHROPIC_API_KEY` from the environment automatically — that is why setting it in your operating system once is enough.

In [ ]:
# Import the official anthropic Python SDK.
import anthropic

# Create the Anthropic client; it picks up ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

# Print a confirmation that the client object was created successfully.
print("Anthropic client ready.")

### Step 3 — our first call: ask Claude for an EDA checklist

For our first message we will ask Claude something simple but practically useful: *"For data like ours, what is a beginner-friendly checklist of EDA steps to perform?"* This is a good first call because:

- The answer does not depend on data we have not loaded.
- The output is a short list, easy to read.
- There is no risk of "hallucinated" numbers — we are not asking Claude to *compute* anything.

Here is the structure of every API call you will make:

- **`model`** — which Claude variant to use. We use **`claude-haiku-4-5`** today: the smallest, fastest, cheapest. Plenty good for short tasks like writing checklists.
- **`max_tokens`** — an upper limit on how long the reply can be. Acts as a safety belt against runaway responses.
- **`system`** — a hidden instruction telling Claude *who it is* and *how to behave* during this call. The user never sees this; the model does.
- **`messages`** — the conversation. For our first call this is just one message from `"role": "user"`.

The reply object has a `.content` attribute which is a list of "content blocks"; for plain-text replies, the first block's `.text` is the message Claude wrote back.

In [ ]:
# Call Claude with one user message; assign the response object to `checklist_response`.
checklist_response = client.messages.create(
    model="claude-haiku-4-5",         # which Claude variant to use (Haiku = small and cheap)
    max_tokens=512,                    # safety cap on the reply length
    system=(                           # hidden persona/instruction shaping Claude's behavior
        "You are a senior data analyst mentoring an absolute beginner. "
        "Speak clearly, avoid jargon, and keep answers concise."
    ),
    messages=[                         # the conversation — just one user message here
        {
            "role": "user",
            "content": (
                "I am a beginner data analyst working with hourly bike-rental data. "
                "Each row is one hour, and I have columns for date, hour of day, weather, "
                "season, and the count of rentals in that hour. "
                "Please give me a short numbered checklist (5 items, one sentence each) "
                "of the first exploratory-data-analysis steps I should perform on a dataset like this."
            )
        }
    ]
)

# Extract Claude's reply text from the response object and print it.
print(checklist_response.content[0].text)

**What we see.** Claude returns a short numbered list of EDA steps tailored to our data. The exact wording will vary between runs (Claude is generative — the same prompt does not always give exactly the same output), but the items will line up with what we already did in sections 1.4–1.6: check shape, check missing values, plot distributions, look at time-of-day patterns, look at weather effects.

> **Why is this useful?** Even if you already know what to do, asking Claude for a checklist *first* is a great way to sanity-check yourself. If Claude's list contains a step you skipped, that is a useful nudge.

### Step 4 — second call: translate our numbers into a stakeholder paragraph

Now the more interesting use. Our `describe()` output is a wall of numbers. Leadership does not want numbers — they want a sentence or two of plain English. We will:

1. Compute the numbers in Python (we already have them).
2. Hand the numbers to Claude.
3. Ask Claude to write a short paragraph an operations manager could read.

This is the **"Python computes, the model interprets"** pattern — one of the most important habits in this course. **Never** let the model do the math; **always** let the model do the translation.

In [ ]:
# Build a small text summary of the numbers we ALREADY computed in Python; we will hand this string to Claude.
stat_summary = (
    f"Two years of hourly bike-rental data (17,379 hours total).\n"
    f"Mean rentals per hour:   {df['cnt'].mean():.2f}\n"
    f"Median rentals per hour: {df['cnt'].median():.2f}\n"
    f"Standard deviation:      {df['cnt'].std():.2f}\n"
    f"Min: {df['cnt'].min()}, Max: {df['cnt'].max()}\n"
    f"Standard error of the overall mean: {df['cnt'].std() / np.sqrt(len(df)):.2f}"
)

# Print a heading so it is clear what is being shown next.
print("Numbers we are sending to Claude:\n")

# Print the summary string itself so you can see exactly what Claude is given as input.
print(stat_summary)

In [ ]:
# Call Claude a second time, this time asking it to turn the computed numbers into one paragraph of plain English.
interpretation_response = client.messages.create(
    model="claude-haiku-4-5",         # same Haiku model — small and cheap
    max_tokens=400,                    # short paragraph; cap reply length
    system=(                           # explicit instruction: no jargon, no inventing numbers
        "You translate analysis numbers into one short paragraph aimed at an operations manager. "
        "No jargon. No mathematical symbols. Use only the numbers given to you; do not invent new numbers."
    ),
    messages=[                         # one user message containing the stats and the writing task
        {
            "role": "user",
            "content": (
                "Here are the basic statistics from my bike-rental dataset:\n\n"
                + stat_summary
                + "\n\nWrite ONE short paragraph (3 to 5 sentences) explaining what these numbers mean "
                + "for someone planning daily operations. Make sure to mention both the typical demand "
                + "and the level of uncertainty."
            )
        }
    ]
)

# Print Claude's stakeholder-paragraph reply.
print(interpretation_response.content[0].text)

**What we see.** Claude writes a short, plain-English paragraph that uses the numbers *we* computed (mean, median, spread) and translates them into operations language.

A few important observations about this call:

- We did **not** ask Claude to compute anything. We computed in Python and *passed the numbers in* through the prompt.
- The `system` message explicitly tells Claude **not to invent numbers**. This is a safety habit — language models can confidently produce plausible-sounding numbers that are wrong. Restricting the model to the numbers we provide is one of the simplest ways to keep it honest.
- The reply costs a tiny amount of credit — a fraction of a US cent. You can watch your usage on `https://platform.claude.com/dashboard`.

> **Mini-recap.** Two API calls. The first asked Claude for help with *process* (an EDA checklist). The second asked Claude for help with *communication* (translating computed numbers into stakeholder language). Neither call asked Claude to do math. That separation — **compute in Python, interpret with the model** — is the habit we build for the rest of the course.

***
## 1.12 Stakeholder memo — translating analysis into a decision

It is Friday afternoon. Time to deliver the memo to leadership.

A good analyst's memo answers four questions, in order:

1. **What did I look at?** — set the scope.
2. **What did I find?** — the headlines, in plain language.
3. **What do I recommend?** — at least one concrete next action.
4. **What do I not yet know?** — honest acknowledgement of limits and uncertainty.

This four-part structure forces *honesty* into your communication. You cannot skip the limitations section, and you cannot recommend something without first showing what you found. Use this same template every week of this course.

### A fully worked example for CityCycle

Below is the memo you would hand to leadership at CityCycle. Read it carefully and notice that **every claim in the memo is grounded in something we computed in this notebook**.

---

**To:** CityCycle Operations Leadership
**From:** [Your name], Junior Analyst
**Re:** Demand patterns and uncertainty — first findings
**Date:** Friday, end of week 1

#### 1. What I looked at
Two years of hourly bike-rental records — 17,379 hours in total — including time of day, day of week, weather, and total rentals per hour.

#### 2. What I found
- **Demand is strongly bimodal across the day.** Two daily peaks dominate: a smaller morning peak around 8 a.m. and a larger evening peak around 5–6 p.m. Mid-day is moderate; overnight is essentially empty. The shape is consistent across the two years of data.
- **A typical hour rents about 142 bikes (median); the average is pulled up to ~189 by the busiest peaks.** Half of all hours fall below 142 rentals; the busiest 25% of hours exceed 281.
- **Hour-to-hour variation is large.** The standard deviation is ~181 — comparable in size to the mean itself. Planning for a single "average" hour will leave the system understaffed at peaks and overstaffed in valleys.
- **Weather has a clear but second-order effect.** Light mist barely affects rentals; light precipitation reduces them noticeably; heavy weather is rare but suppresses demand further.
- **The reported overall mean is trustworthy.** Because it is built on 17,379 hours, its standard error is approximately 1.4 — well under 1% of the mean. The number "189 rentals per hour, on average" can be reported with confidence.

#### 3. What I recommend
- **Drive staffing and bike-rebalancing schedules from hourly demand curves, not from a single daily average.** The mid-day average obscures the peaks where the system actually needs attention.
- **Treat the 5–6 p.m. evening peak as the most operationally critical hour.** It accounts for the largest spike in demand and the highest risk of empty stations.
- **Monitor weather forecasts and proactively reduce rebalancing effort on poor-weather days.** Demand drops are predictable enough to plan around.

#### 4. What I do not yet know
- This data does not include **station-level information**; recommendations about *where* to rebalance bikes need station-level data we do not have here.
- The "weather effect" is averaged across all hours and seasons; it likely interacts strongly with time of day (a rainy 5 p.m. is not the same as a rainy 3 a.m.). A follow-up analysis should cross weather with hour.
- The data set ends in 2012; **a refresh with current data** is needed before any of these recommendations are operationalized at today's scale.

---

That is your memo. In Session 02 we will tighten it further and turn it into a more polished risk-style brief. For now, **save your own version as `_reports/session01_memo.md`** in your course directory. The instructor will review it next week.

***
## 1.13 References — what to study to deepen this session

You can finish this session without watching anything else, but if you want each idea to *stick*, pick two or three videos from the list below and watch them before Session 02. Each StatQuest video is short (under 15 minutes) and clarifies exactly one concept from today. Khan Academy lessons are usually slightly longer and pair the video with practice exercises on their main site.

### StatQuest videos (YouTube)

| Video | What it clarifies |
|---|---|
| [Histograms, Clearly Explained](https://www.youtube.com/watch?v=qBigTkBLU6g) | The single most important plot in introductory data analysis: what a histogram is, how it is built from raw data, and how the bin width changes what you see. Direct companion to §1.6 Plot 1. |
| [The Main Ideas behind Probability Distributions](https://www.youtube.com/watch?v=oI3hZJqXJuc) | Demystifies the very idea of a probability distribution at the level of intuition — no formulas. The natural follow-up to §1.7's "Where do shapes come from?" cell. |
| [The Normal Distribution, Clearly Explained](https://www.youtube.com/watch?v=rzFX5NWojp0) | The bell curve from scratch: how it is generated, how its mean and standard deviation should be interpreted, why it shows up everywhere. Pairs with §1.7 Story 1. |
| [Population and Estimated Parameters](https://www.youtube.com/watch?v=vikkiwjQqfU) | The sample-vs-population distinction from §1.8 made visual: what a population parameter is, and what it really means to *estimate* one from a sample. |
| [Calculating the Mean, Variance and Standard Deviation](https://www.youtube.com/watch?v=SzZ6GpcfoQY) | The arithmetic behind §1.5 step-by-step — including the very common pitfall about which divisor (n versus n − 1) to use, and why it matters. |
| [Sampling from a Distribution](https://www.youtube.com/watch?v=XLCWeSVzHUU) | Short and to the point (under 4 minutes): what does it actually mean to "draw a sample" from a distribution? Useful background for the simulations in §1.8 and §1.9. |
| [The Binomial Distribution and Test](https://www.youtube.com/watch?v=J8jNoF-K8E8) | A deeper dive into the "yes/no count" story from §1.7, plus the basic statistical test that goes with it. |
| [The Central Limit Theorem, Clearly Explained](https://www.youtube.com/watch?v=YAlJCEDH2uY) | The single most important video on this list. About 8 minutes that crystallize §1.9: where the bell-shaped sampling distribution of the mean comes from and why it is the foundation of honest business reporting. |
| [Standard Deviation vs Standard Error](https://www.youtube.com/watch?v=A82brFpdr9g) | The exact distinction from §1.10 — the one that, when missed, causes embarrassing business-report mistakes. Pairs directly with our "two kinds of wobble" framing. |
| [The Standard Error](https://www.youtube.com/watch?v=XNgt7F6FqDU) | A second angle on the same idea, with a bootstrap-based explanation of how to *compute* the standard error empirically (rather than via the SD/√n formula). |

### Khan Academy resources (YouTube videos and practice site)

| Resource | What it clarifies |
|---|---|
| [Statistics & Probability hub](https://www.khanacademy.org/math/statistics-probability) (website, not YouTube) | Khan Academy's structured introductory statistics course. Treat it as a reference and use the **practice exercises** on the site to reinforce each concept after you watch a video. The site, not YouTube, is where Khan really shines. |
| [How to interpret a histogram](https://www.youtube.com/watch?v=c02vjunQsJM) | A short, slow walk-through of reading a histogram — what the bins are, what each bar's height represents. A gentle companion to §1.6 Plot 1. |
| [Interpreting box plots](https://www.youtube.com/watch?v=oBREri10ZHk) | Reads a box plot piece-by-piece: median, the two quartiles, whiskers, outliers. Pair with §1.6 Plot 2. |
| [Range, variance and standard deviation as measures of dispersion](https://www.youtube.com/watch?v=E4HAYd0QnRc) | The three classic spread measures from §1.5.4, computed by hand on a small example. A useful refresher if the variance formula still feels intimidating. |
| [Introduction to the normal distribution](https://www.youtube.com/watch?v=hgtMWR3TFnY) | Khan's introduction to the bell curve, density curves, and reading areas underneath them. Complements StatQuest's Normal video — pick whichever style suits you better. |
| [Central limit theorem](https://www.youtube.com/watch?v=JNm3M9cqWyc) | Khan's version of the §1.9 idea. Slower and more deliberate than StatQuest's; especially good if you want to follow up with the *"Sample means and the central limit theorem"* practice exercises on the Khan Academy site. |

> A good weekly rhythm: **two StatQuest videos for intuition, plus one Khan Academy practice unit for hands-on reinforcement.** StatQuest is the "intuition channel"; Khan Academy is the "practice space".

See you in **Session 02**, where we move from descriptive statistics into *probability* — and you will use what you learned today to talk honestly about risk.

<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)

<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>